# Requirement & Setup

In [ ]:
#!pip install https://github.com/scikit-learn/sklearn-pypi-package/releases/download/1.3.0/sklearn-1.3.0-py2.py3-none-any.whl

In [ ]:
#!pip install kagglehub

In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from tqdm import tqdm
import kagglehub

In [ ]:
dataset_path = kagglehub.competition_download('super-ai-engineer-ss-6-sleep-stage-classification')

TRAIN_FOLDER = os.path.join(dataset_path, "train", "train")
TEST_FOLDER  = os.path.join(dataset_path, "test_segment", "test_segment")
SUB_PATH     = os.path.join(dataset_path, "sample_submission.csv")
OUTPUT_SUB_PATH = "submission_crnn_optimized.csv"

if not os.path.exists(TRAIN_FOLDER):
    TRAIN_FOLDER = os.path.join(dataset_path, "train")
if not os.path.exists(TEST_FOLDER):
    TEST_FOLDER = os.path.join(dataset_path, "test_segment")

FEATURES = ['BVP', 'ACC_X', 'ACC_Y', 'ACC_Z', 'TEMP', 'EDA', 'HR', 'IBI']
SEQ_LENGTH = 480
BATCH_SIZE = 16
EPOCHS = 30
LEARNING_RATE = 0.001
PATIENCE = 7

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

# data

In [ ]:
def load_train_data(train_dir):
    all_x, all_y = [], []
    
    search_pattern = os.path.join(train_dir, "**", "*.csv")
    file_list = glob.glob(search_pattern, recursive=True)

    if len(file_list) == 0:
        raise FileNotFoundError(f"❌ ไม่พบไฟล์ .csv เลยในโฟลเดอร์ {train_dir} หรือโฟลเดอร์ย่อย ลองเช็ค Path ใน Cell 3 ดูอีกครั้งครับ")
    
    for file in tqdm(file_list, desc=f"Reading {len(file_list)} Train Files"):
        df = pd.read_csv(file)
        n_segments = len(df) // SEQ_LENGTH
        
        if n_segments == 0:
            continue
        
        df = df.iloc[:n_segments * SEQ_LENGTH]
        
        X_data = df[FEATURES].values
        Y_data = df['Sleep_Stage'].values
        
        X_reshaped = X_data.reshape(n_segments, SEQ_LENGTH, len(FEATURES))
        Y_reshaped = Y_data[::SEQ_LENGTH]
        
        all_x.append(X_reshaped)
        all_y.append(Y_reshaped)
        
    if not all_x:
         raise ValueError("❌ เจอไฟล์ .csv แต่ไม่มีไฟล์ไหนที่มีข้อมูลยาวพอ (480 แถว) เลย")
         
    return np.concatenate(all_x, axis=0), np.concatenate(all_y, axis=0)

In [ ]:
def load_test_data(test_dir, sample_sub_path):
    sub_df = pd.read_csv(sample_sub_path)
    test_ids = sub_df['id'].values
    all_x = []
    
    for tid in tqdm(test_ids, desc="Reading Test Segments"):
        folder_name = str(tid).split('_')[0]
        file_path = os.path.join(test_dir, folder_name, f"{tid}.csv")
        
        if not os.path.exists(file_path):
            all_x.append(np.zeros((SEQ_LENGTH, len(FEATURES))))
            continue
            
        df = pd.read_csv(file_path)
        X_data = df[FEATURES].values
        
        if len(X_data) < SEQ_LENGTH:
            last_row = X_data[-1, :]
            pad = np.tile(last_row, (SEQ_LENGTH - len(X_data), 1))
            X_data = np.vstack([X_data, pad])
        else:
            X_data = X_data[:SEQ_LENGTH]
        all_x.append(X_data)
    return np.array(all_x), sub_df

In [ ]:
class SleepDataset(Dataset):
    def __init__(self, X, Y=None, augment=False):
        self.X = torch.tensor(X, dtype=torch.float32).transpose(1, 2)
        self.Y = torch.tensor(Y, dtype=torch.long) if Y is not None else None
        self.augment = augment

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = self.X[idx]
        
        if self.augment:
            noise = torch.randn_like(x) * 0.02
            x = x + noise
            
        if self.Y is not None: 
            return x, self.Y[idx]
        return x

# Model

In [ ]:
class EarlyStopping:
    def __init__(self, patience=5, path='best_model.pth'):
        self.patience = patience
        self.path = path
        self.counter = 0
        self.best_loss = np.inf
        self.early_stop = False

    def __call__(self, val_loss, model):
        if val_loss < self.best_loss:
            self.best_loss = val_loss
            self.counter = 0
            torch.save(model.state_dict(), self.path)
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True

In [ ]:
class Attention(nn.Module):
    def __init__(self, hidden_size):
        super(Attention, self).__init__()
        self.attention = nn.Linear(hidden_size, 1)

    def forward(self, lstm_out):
        attn_weights = self.attention(lstm_out)
        attn_weights = torch.softmax(attn_weights, dim=1)
        
        context = torch.sum(attn_weights * lstm_out, dim=1)
        return context, attn_weights

class SleepStageCRNN(nn.Module):
    def __init__(self, num_features, num_classes):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv1d(num_features, 64, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2),
            
            nn.Conv1d(64, 128, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(2)
        )
        
        self.lstm = nn.LSTM(input_size=128, hidden_size=64, num_layers=2, 
                            batch_first=True, dropout=0.3, bidirectional=True)
        
        self.attention = Attention(hidden_size=128)
        
        self.fc = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        c = self.cnn(x)                     
        c = c.transpose(1, 2)               
        lstm_out, _ = self.lstm(c)          
            
        context, _ = self.attention(lstm_out) 
        
        out = self.fc(context)             
        return out

# preparation / class weight

In [ ]:
X_train_raw, Y_train_raw = load_train_data(TRAIN_FOLDER)

In [ ]:
display(X_train_raw)

In [ ]:
display(Y_train_raw)

In [ ]:
scaler = StandardScaler()
N_tr, T, F = X_train_raw.shape
X_train_scaled = scaler.fit_transform(X_train_raw.reshape(-1, F)).reshape(N_tr, T, F)

In [ ]:
label_encoder = LabelEncoder()
Y_train_enc = label_encoder.fit_transform(Y_train_raw)
num_classes = len(label_encoder.classes_)
print(f"Classes Detected: {label_encoder.classes_}")

class_weights_np = compute_class_weight('balanced', classes=np.unique(Y_train_enc), y=Y_train_enc)
class_weights = torch.tensor(class_weights_np, dtype=torch.float).to(device)
print(f"Class Weights: {class_weights_np}")

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X_train_scaled, Y_train_enc, test_size=0.2, random_state=42, stratify=Y_train_enc
)

train_dataset = SleepDataset(X_train, y_train, augment=True)
val_dataset = SleepDataset(X_val, y_val, augment=False)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, pin_memory=True)

# train

In [ ]:
model = SleepStageCRNN(num_features=len(FEATURES), num_classes=num_classes).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights) 
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

early_stopping = EarlyStopping(patience=PATIENCE, path='best_sleep_model.pth')

In [ ]:
for epoch in range(EPOCHS):
    model.train()
    train_loss, correct, total = 0.0, 0, 0
    
    for batch_x, batch_y in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]"):
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        
        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += batch_y.size(0)
        correct += (predicted == batch_y).sum().item()
        
    train_acc = 100 * correct / total
    
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for batch_x, batch_y in val_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            
            val_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            val_total += batch_y.size(0)
            val_correct += (predicted == batch_y).sum().item()
            
    val_acc = 100 * val_correct / val_total
    avg_val_loss = val_loss / len(val_loader)
    
    scheduler.step(avg_val_loss)
    current_lr = optimizer.param_groups[0]['lr']
    
    print(f"Train Loss: {train_loss/len(train_loader):.4f} | Train Acc: {train_acc:.2f}%")
    print(f"Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.2f}% | LR: {current_lr}")
    
    # ตรวจสอบ Early Stopping
    early_stopping(avg_val_loss, model)
    if early_stopping.early_stop:
        print(f"Early stopping triggered at epoch {epoch+1}")
        break

# submission

In [ ]:
model.load_state_dict(torch.load('best_sleep_model.pth'))
model.eval()

In [ ]:
X_test_raw, sub_df = load_test_data(TEST_FOLDER, SUB_PATH)

N_te, _, _ = X_test_raw.shape
X_test_scaled = scaler.transform(X_test_raw.reshape(-1, F)).reshape(N_te, T, F)

test_dataset = SleepDataset(X_test_scaled, augment=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, pin_memory=True) 
all_preds = []

with torch.no_grad():
    for batch_x in tqdm(test_loader, desc="Predicting"):
        batch_x = batch_x.to(device)
        outputs = model(batch_x)
        _, predicted = torch.max(outputs, 1)
        all_preds.extend(predicted.cpu().numpy())
        
decoded_preds = label_encoder.inverse_transform(all_preds)

sub_df['labels'] = decoded_preds
sub_df.to_csv(OUTPUT_SUB_PATH, index=False)

print(f"\nDone! Submission file successfully saved to:\n{OUTPUT_SUB_PATH}")